# Dental Caries Detection — Colab Training (1280px, 3-class)

Trains the yolov8m dental caries detector at 1280px resolution on Colab's GPU.

**Before running:** upload `data_for_colab.zip` (from the project's `data_for_colab.zip`) to the root of your Google Drive ("My Drive").

## 1. Check GPU and install dependencies

In [ ]:
!nvidia-smi
!pip install -q ultralytics

## 2. Mount Drive and unzip the dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/data_for_colab.zip /content/
!unzip -q /content/data_for_colab.zip -d /content/
!ls /content/data

## 3. Fix data.yaml (the zip's copy has a local Windows path baked in)

In [ ]:
data_yaml = """path: /content/data
train: images/train
val: images/val
nc: 3
names: ['Impacted', 'Caries', 'Deep Caries']
"""
with open('/content/data/data.yaml', 'w') as f:
    f.write(data_yaml)

!cat /content/data/data.yaml

## 3b. Check actual CPU core count

Colab's free tier typically gives only ~2 vCPUs. The default dataloader `workers=8` badly oversubscribes that — that mismatch is why the T4 sat at 0% GPU-util and 29W/70W power draw in earlier testing (the GPU was starved waiting on CPU-side image decode/resize, not compute-limited). Run this to confirm your actual core count before training.

In [ ]:
import os
print("CPU cores available:", os.cpu_count())

## 3c. Start a background GPU monitor

Colab runs one cell at a time — you can't run `!nvidia-smi` in another cell while training blocks the kernel. This starts `nvidia-smi` as a background OS **subprocess** instead (not a notebook cell), logging GPU utilization/power every 5 seconds to a file. It keeps running independently of the training cell, so you get a full history to check afterward (or if you interrupt training) rather than a single snapshot.

In [ ]:
import subprocess

monitor_proc = subprocess.Popen(
    "nvidia-smi --query-gpu=timestamp,utilization.gpu,power.draw,memory.used "
    "--format=csv -l 5 > /content/gpu_monitor.log",
    shell=True,
)
print("GPU monitor started in the background (PID:", monitor_proc.pid, ")")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")
model.train(
    data="/content/data/data.yaml",
    epochs=100,
    imgsz=1280,
    batch=-1,
    workers=2,
    cache='disk',
    patience=20,
    name="dental_caries_demo_m_3class_hires_1280_colab",
    project="runs/detect",
)

In [ ]:
!tail -40 /content/gpu_monitor.log

## 4b. Check GPU utilization history

Run this any time — including by clicking the training cell's stop (■) button first if you don't want to wait for it to finish. Interrupting is safe: Ultralytics checkpoints `last.pt` after each completed epoch, and you can resume later.

Read the `utilization.gpu` column: if it's consistently near 90-100%, the T4 itself is the bottleneck (Colab Pro's faster GPUs would help). If it's low/bouncing around, something is still starving it before compute.

## 5. Download the trained weights

In [ ]:
from google.colab import files
files.download('/content/runs/detect/dental_caries_demo_m_3class_hires_1280_colab/weights/best.pt')

## Notes

- Colab's free tier disconnects after ~90 minutes of tab inactivity and has a ~12 hour hard session cap. Keep the tab open/active, or use Colab Pro for a more reliable long run.
- If the session disconnects mid-training, Ultralytics saves checkpoints under `runs/detect/.../weights/last.pt` each epoch — remount Drive, re-copy/unzip the data, and resume with `model = YOLO('runs/detect/.../weights/last.pt'); model.train(resume=True)`.
- This dataset (DENTEX + a Kaggle cavity set + CariesXrays) is a mix of research-only and unlicensed sources — see the project's CLAUDE.md. Local demo/prototyping use only, not for redistribution.